# Lab 4 — Cloudera Specifics

**Objective:** Understand two CDW-specific capabilities — Iceberg table format and Ranger-based security — that apply to every workload you build on the platform.

**Estimated time:** ~30 minutes

**Prerequisites:** Lab 1 completed successfully (`iceberg.airline_lab.flights` is populated).

---

## What you will do

### Part 1 — Iceberg Table Format
CDW treats Apache Iceberg as a first-class table format. You will exercise four features directly:

| Feature | What happens |
|---|---|
| **Schema evolution** | `ALTER TABLE … ADD COLUMN` without rewriting data. Old readers still see the pre-evolution schema. |
| **Hidden partitioning** | The table is partitioned by `year(departure_date)`. Queries with a `departure_date` predicate prune partitions automatically — you do not filter on a partition column. |
| **Time travel** | `SELECT * FROM flights FOR VERSION AS OF <snapshot_id>` or `… FOR TIMESTAMP AS OF TIMESTAMP '…'` |
| **Safe rollback** | `CALL system.rollback_to_snapshot(…)` undoes a bad write without restoring from backup. |

### Part 2 — Ranger Security Integration
Every query Trino runs in CDW goes through Apache Ranger for authorization:
- Your workload user needs an explicit `SELECT` (or wider) policy on the schema.
- Column-level masking and row-level filters are enforced **server-side** — no client change required.
- Every query is audited. Ranger's **Audits** tab is the authoritative trail.

**Verification:** At least one time-travel query returns a result, and the Ranger audit log shows your queries.

---
## Setup — Connection

In [ ]:
import os
import pandas as pd
from trino.dbapi import connect
from trino.auth import BasicAuthentication

conn = connect(
    host=os.environ['TRINO_HOST'],
    port=int(os.environ.get('TRINO_PORT', 443)),
    user=os.environ['TRINO_USER'],
    auth=BasicAuthentication(os.environ['TRINO_USER'], os.environ['TRINO_PASSWORD']),
    http_scheme='https',
    catalog='iceberg',
    schema='airline_lab',
)

def q(sql):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    return pd.DataFrame(rows, columns=cols)

def execute(sql):
    cur = conn.cursor()
    cur.execute(sql)
    cur.fetchall()
    print('OK')

# Sanity check
q('SELECT COUNT(*) AS total_flights FROM iceberg.airline_lab.flights')

---
## Part 1 — Iceberg Table Format

### Feature 1 — Schema Evolution

Iceberg tracks schema versions independently of the data files.  
`ALTER TABLE … ADD COLUMN` is instantaneous — no data is rewritten.  
Existing rows return `NULL` for the new column; old readers that don't know the column are unaffected.

#### Add a new column — no data rewrite, completes in milliseconds

In [ ]:
execute("ALTER TABLE iceberg.airline_lab.flights ADD COLUMN loyalty_tier VARCHAR")

#### Verify the new column appears in the schema

In [ ]:
q("DESCRIBE iceberg.airline_lab.flights")

#### Existing rows return NULL for loyalty_tier — backward compatible, no data touched

In [ ]:
q("""
    SELECT passenger_id, first_name, flight_status, loyalty_tier
    FROM iceberg.airline_lab.flights
    LIMIT 5
""")

Note: existing rows can be backfilled anytime using a simple UPDATE statement

### Feature 2 — Hidden Partitioning

The `flights` table is partitioned by `year(departure_date)`.  
Iceberg stores the partition transform in the table metadata — **you never filter on a synthetic partition column**.  
When a query has a `departure_date` predicate, Trino reads only the matching yearly partition files.  
Use `EXPLAIN` to confirm partition pruning is happening.

#### Query with a date predicate — Trino prunes to the 2022 partition automatically

In [ ]:
q("""
    SELECT flight_status, COUNT(*) AS total
    FROM iceberg.airline_lab.flights
    WHERE departure_date BETWEEN DATE '2022-01-01' AND DATE '2022-12-31'
    GROUP BY flight_status
    ORDER BY total DESC
""")

#### EXPLAIN confirms partition pruning

In [ ]:
cur = conn.cursor()
cur.execute("""
    EXPLAIN SELECT COUNT(*)
    FROM iceberg.airline_lab.flights
    WHERE departure_date BETWEEN DATE '2022-01-01' AND DATE '2022-12-31'
""")
print(cur.fetchone()[0])

### Feature 3 — Time Travel

Every write (`INSERT`, `UPDATE`, `DELETE`, `OPTIMIZE`) creates a new Iceberg **snapshot**.  
You can query any historical snapshot by id or by timestamp — no backup or restore required.  

The `flights$snapshots` metadata table lists every snapshot with its id, timestamp, and operation.

#### List all snapshots — note the snapshot_id and committed_at values
 Each row is a point-in-time version of the table.
 'operation' tells you what caused the snapshot: append, overwrite, delete etc.
 You should see at least two rows:
   - First append  → empty table (0 rows) — created when the table was defined
   - Second append → data loaded  — created when INSERT ran in Lab 1

In [ ]:

snaps = q("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg.airline_lab."flights$snapshots"
    ORDER BY committed_at
""")
snaps

#### Time travel by snapshot_id
`FOR VERSION AS OF` <snapshot_id> tells Trino to read the table exactly as it was when that snapshot was created.

This is how you would audit "what did the table look like before that bad INSERT ran?" in a real production scenario.

In [ ]:
first_snap = int(snaps.iloc[0]['snapshot_id'])
latest_snap = int(snaps.iloc[-1]['snapshot_id'])

print(f'First snapshot : {first_snap}')
display(q(f'SELECT COUNT(*) AS rows_at_first_snapshot FROM iceberg.airline_lab.flights FOR VERSION AS OF {first_snap}'))

print(f'Latest snapshot: {latest_snap}')
display(q(f'SELECT COUNT(*) AS rows_at_latest_snapshot FROM iceberg.airline_lab.flights FOR VERSION AS OF {latest_snap}'))

#### Time travel by timestamp — same result, different syntax

`FOR TIMESTAMP AS OF TIMESTAMP` '...' does the same thing `as FOR VERSION AS OF` but uses a wall-clock time instead of a snapshot id.
This is more practical in real incidents — you usually know "the bad job ran at 3pm" rather than the exact snapshot id.
We use the committed_at time of each snapshot from the table above

In [ ]:

first_ts = snaps.iloc[0]['committed_at']
latest_ts = snaps.iloc[-1]['committed_at']

print(f'Travelling to: {first_ts}')
display(q(f"SELECT COUNT(*) AS rows_at_timestamp FROM iceberg.airline_lab.flights FOR TIMESTAMP AS OF TIMESTAMP '{first_ts}'"))

print(f'Travelling to: {latest_ts}')
display(q(f"SELECT COUNT(*) AS rows_at_timestamp FROM iceberg.airline_lab.flights FOR TIMESTAMP AS OF TIMESTAMP '{latest_ts}'"))

### Feature 4 — Safe Rollback

`CALL system.rollback_to_snapshot(…)` resets the table's current snapshot pointer — **no data is physically deleted or restored**.  
This is the recommended recovery path for accidental writes in CDW: faster than restoring from S3 backup and fully atomic.

In the cells below you will:
1. Record the current (clean) snapshot id.
2. Simulate a bad `UPDATE` that corrupts data.
3. Roll back to the clean snapshot.
4. Verify the data is restored.

#### Record the clean snapshot before making any changes

In [ ]:
snaps_before = q("""
    SELECT snapshot_id, committed_at, operation
    FROM iceberg.airline_lab."flights$snapshots"
    ORDER BY committed_at
""")
clean_snap = int(snaps.iloc[-1]['snapshot_id'])
print(f'Clean snapshot to roll back to: {clean_snap}')

# Current row counts by status
q("SELECT flight_status, COUNT(*) AS cnt FROM iceberg.airline_lab.flights GROUP BY flight_status ORDER BY cnt DESC")

#### Simulate a bad write (corrupt all 'On Time' records)

In [ ]:

execute("""
    UPDATE iceberg.airline_lab.flights
    SET flight_status = 'CORRUPTED'
    WHERE flight_status = 'On Time'
""")

# Confirm the damage
q("SELECT flight_status, COUNT(*) AS cnt FROM iceberg.airline_lab.flights GROUP BY flight_status ORDER BY cnt DESC")

#### Roll back to the clean snapshot

In [ ]:
execute(f"CREATE OR REPLACE TABLE iceberg.airline_lab.flights AS SELECT * FROM iceberg.airline_lab.flights FOR VERSION AS OF {clean_snap}")

#### Verify CORRUPTED rows are gone and On Time records are restored

In [ ]:
q("SELECT flight_status, COUNT(*) AS cnt FROM iceberg.airline_lab.flights GROUP BY flight_status ORDER BY cnt DESC")

---
## Part 2 — Ranger Security Integration

Every query Trino runs in CDW is authorized by **Apache Ranger** before execution.

### How it works

- **Policy-based access control:** Your workload user needs an explicit `SELECT` (or wider) Ranger policy on the target catalog → schema → table.
- **Column-level masking:** A Ranger masking policy can hash or redact a column (e.g. `first_name`) server-side. The client receives the masked value — no query change required.
- **Row-level filtering:** A Ranger row filter policy restricts which rows a user sees, again enforced server-side.
- **Audit trail:** Every query — allowed or denied — is written to Ranger's audit log. The **Audits** tab in the Ranger UI is the authoritative record.

### Common error

```
Access Denied: Cannot select from table iceberg.airline_lab.flights
```
→ Request a Ranger policy granting `SELECT` on `iceberg → airline_lab → flights` for your user or group.

### Steps to verify Ranger is active

This query succeeds only if Ranger has granted SELECT to your user.
If it throws 'Access Denied', request a access policy for your user from your administrator.

In [ ]:
q("""
    SELECT first_name, last_name, nationality
    FROM iceberg.airline_lab.flights
    LIMIT 5
""")

### Column masking — what to expect

If a Ranger **masking policy** is applied to `first_name` (e.g. a SHA-256 hash rule), the query above will return hashed values instead of plain names — even though you wrote `SELECT first_name`.  
The masking is enforced server-side; the client code does not change.

To configure this in Ranger:
1. Open **Ranger UI → Trino service → Masking Policies**.
2. Add a policy for `iceberg → airline_lab → flights → first_name`.
3. Set masking type to **Redact** for the `workshop_user` group.
4. Re-run the cell above — `first_name` will now return hashed values.

In [ ]:
# Run after applying a Ranger masking policy on first_name.
# Expected: first_name returns a hash string, last_name returns plaintext.
q("""
    SELECT first_name, last_name
    FROM iceberg.airline_lab.flights
    LIMIT 10
""")

### Audit log verification

After running the queries above:
1. Open **Ranger UI → Audit → Access**.
2. Filter by **User** = your workload user and **Service_Name** = cm_trino.
3. You should see one audit entry per query, with result `Allowed` or `Denied`.

This confirms that every query — regardless of whether it touched sensitive columns — is logged by Ranger.

---
## Lab 4 Summary

| Feature | What you did | Key takeaway |
|---|---|---|
| Schema evolution | Added `loyalty_tier` column | Zero data rewrite; old rows return `NULL` |
| Hidden partitioning | Queried with date predicate + `EXPLAIN` | Trino prunes partitions automatically |
| Time travel | Queried by snapshot id and timestamp | Any historical state queryable without backup |
| Safe rollback | Corrupted data, then rolled back | Atomic recovery in seconds via snapshot pointer |
| Ranger security | `SELECT` gated by policy; masking applied | Authorization and audit fully server-side |